# MEModel current clamp with morphology locations

This notebook shows how to target a current clamp to specific morphology locations in OBI-One.

In this example, we:

1. Define morphology locations on dendritic sections.
2. Attach those locations to a current clamp stimulus.
3. Generate and run the simulation.
4. Inspect the generated files for verification.

OBI-One handles the conversion to simulator-ready targeting internally during simulation generation.

## Imports and database client

Adjust the authentication/environment lines to match your local setup.

In [ ]:
from obi_auth import get_token

import obi_one as obi
from entitysdk import Client, ProjectContext


obi_one_api_url = "http://127.0.0.1:8100"

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=obi.LAB_ID_STAGING_TEST, project_id=obi.PROJECT_ID_STAGING_TEST)
db_client = Client(api_url="https://staging.openbraininstitute.org/api/entitycore", project_context=project_context, token_manager=token)

## Select the MEModel with synapses entity

Replace `entity_ID` with the ID of the MEModel-with-synapses circuit you want to simulate.

In [2]:
entity_ID = "5c43e35b-7e44-4705-a2a1-2e5ac24af536"
sim_duration = 1500.0

## Create the simulation form

In [3]:
sim_form = obi.MEModelWithSynapsesCircuitSimulationScanConfig.empty_config()

info = obi.Info(
    campaign_name="MEModel morphology-location current clamp",
    campaign_description=(
        "Example showing MorphologyLocations materialized into SONATA compartment_sets."
    ),
)
sim_form.set(info, name="info")

## Add timestamps

The current clamp will be applied once at the beginning of the simulation.

In [4]:
timestamps = obi.RegularTimestamps(
    start_time=0.0,
    number_of_repetitions=1,
    interval=sim_duration,
)
sim_form.add(timestamps, name="Timestamps")

## Add morphology locations

`RandomMorphologyLocations` is the user-facing rule for non-somatic subcellular targeting.

Here we ask for two random locations on basal dendrite (`3`) or apical dendrite (`4`) sections.

Section types available:

```text
2 = axon
3 = basal dendrite
4 = apical dendrite

In [5]:
locations = obi.RandomMorphologyLocations(
    random_seed=0,
    number_of_locations=20,
    section_types=(3,4),
)
sim_form.add(locations, name="ClampLocations")

## Add a location-targeted current clamp

`locations` is optional. If omitted, the current clamp targets the default somatic location; if provided, OBI-One materializes it into a SONATA compartment set.

For MEModel-with-synapses, the simulated cell is implicit, so we do not pass a neuron set here.

In [6]:
clamp = obi.ConstantCurrentClampStimulus(
    timestamps=timestamps.ref,
    duration=500.0,
    amplitude=0.5,
    locations=locations.ref,
)
sim_form.add(clamp, name="LocationCurrentClamp")

## Add recording and initialize

In [ ]:
recording = obi.SomaVoltageRecording()
sim_form.add(recording, name="SomaVoltage")

init = obi.MEModelWithSynapsesCircuitSimulationScanConfig.Initialize(
    circuit=obi.MEModelWithSynapsesCircuitFromID(id_str=entity_ID),
    simulation_length=sim_duration,
)
sim_form.set(init, name="initialize")

validated_sim_conf = sim_form.validated_config()

## Generate simulation files

During `run_tasks_for_generated_scan`, the `locations` reference is materialized into a generated `CompartmentSet`.

That means the final SONATA input should reference `compartment_set`, not `locations`.

In [8]:
grid_scan = obi.GridScanGenerationTask(
    form=validated_sim_conf,
    coordinate_directory_option="ZERO_INDEX",
    output_root="./obi-output/morphology_location_current_clamp/grid_scan",
)

grid_scan.execute(db_client=db_client)
obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)

[None]

## Inspect generated SONATA input

The current clamp input should contain a `compartment_set` key.

In [9]:
import json
coordinate_root = grid_scan.single_configs[0].coordinate_output_root
simulation_config_path = coordinate_root / "simulation_config.json"

simulation_config = json.loads(simulation_config_path.read_text())

print(simulation_config_path)
print(json.dumps(simulation_config["inputs"], indent=2))

obi-output/morphology_location_current_clamp/grid_scan/0/simulation_config.json
{
  "LocationCurrentClamp_0": {
    "delay": 0.0,
    "duration": 500.0,
    "compartment_set": "LocationCurrentClamp__locations",
    "module": "linear",
    "input_type": "current_clamp",
    "amp_start": 0.5,
    "represents_physical_electrode": false
  }
}


## Inspect generated compartment set

The `compartment_sets.json` file contains concrete `(node_id, section_id, offset)` entries.

The `section_id` values are generated on morphologies loaded with NEURON-compatible `nrn_order`.

In [16]:
import json
import pandas as pd

coordinate_root = grid_scan.single_configs[0].coordinate_output_root

compartment_sets = json.loads(
    (coordinate_root / "compartment_sets.json").read_text()
)

entries = compartment_sets["LocationCurrentClamp__locations"]["compartment_set"]

circuit = obi.Circuit(
    name="generated_circuit",
    path=str(coordinate_root / "sonata_circuit" / "circuit_config.json"),
)

morph = circuit.load_morphology(0)

SECTION_TYPES = {
    1: "soma",
    2: "axon",
    3: "basal_dendrite",
    4: "apical_dendrite",
}

rows = []
for node_id, section_id, offset in entries:
    section = morph.section(section_id)
    rows.append(
        {
            "node_id": node_id,
            "section_id": section_id,
            "section_type": SECTION_TYPES.get(int(section.type), int(section.type)),
            "offset": offset,
        }
    )

pd.DataFrame(rows)

,node_id,section_id,section_type,offset
0,0,66,basal_dendrite,0.204930
1,0,73,basal_dendrite,0.945059
2,0,78,basal_dendrite,0.666411
3,0,125,basal_dendrite,0.902999
4,0,132,basal_dendrite,0.330224
5,0,133,basal_dendrite,0.890507
6,0,163,apical_dendrite,0.000000
7,0,166,apical_dendrite,0.830645
8,0,168,apical_dendrite,0.057320
9,0,174,apical_dendrite,0.362805


## Optional: run the generated simulation

The cells below are optional and depend on your local NEURON/BlueCelluLab setup.

In [14]:
# circuit_folder = coordinate_root / "sonata_circuit"
# ! rm -rf arm64/
# ! ../../.venv/bin/nrnivmodl -incflags "-DDISABLE_REPORTINGLIB" {circuit_folder}/mod

In [15]:
# from obi_one.scientific.library.simulation.entrypoint import run
#
# run(
#     simulation_config=simulation_config_path,
#     simulator="bluecellulab",
#     libnrnmech_path="./arm64",
# )